# Retrieval-Augmented Generation using Pinecone

This notebook demonstrates how to connect Claude with the data in your Pinecone vector database through a technique called retrieval-augmented generation (RAG). We will cover the following steps:

1. Embedding a dataset using Voyage AI's embedding model
2. Uploading the embeddings to a Pinecone index
3. Retrieving information from the vector database
4. Using Claude to answer questions with information from the database

## Setup
First, let's install the necessary libraries. You will need a [Claude API key](https://console.anthropic.com/), a [Pinecone API key](https://docs.pinecone.io/docs/quickstart), and a [Voyage AI API key](https://docs.voyageai.com/install/). Set them as environment variables: `ANTHROPIC_API_KEY`, `PINECONE_API_KEY`, and `VOYAGE_API_KEY`.

In [1]:
!uv pip install -q "anthropic>=0.40.0" "pinecone>=5.0.0" voyageai pandas tqdm

In [2]:
import os

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")

## Download the dataset
Now let's download the Amazon products dataset which has over 10k Amazon product descriptions and load it into a DataFrame.

In [3]:
import pandas as pd

# Download the JSONL file
!wget  https://www-cdn.anthropic.com/48affa556a5af1de657d426bcc1506cdf7e2f68e/amazon-products.jsonl

data = []
with open("amazon-products.jsonl") as file:
    for line in file:
        try:
            data.append(eval(line))  # noqa: S307
        except (SyntaxError, ValueError):
            # Skip malformed lines in the dataset
            pass

df = pd.DataFrame(data)
display(df.head())
len(df)

zsh:1: command not found: wget


,text
0,Product Name: DB Longboards CoreFlex Crossbow ...
1,Product Name: Electronic Snap Circuits Mini Ki...
2,Product Name: 3Doodler Create Flexy 3D Printin...
3,Product Name: Guillow Airplane Design Studio w...
4,Product Name: Woodstock- Collage 500 pc Puzzle...


10002

## Vector Database

To create our vector database, we first need a free API key from Pinecone. Once we have the key, we can initialize the database as follows:

In [4]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

Next, we set up our index specification, which allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all available providers and regions [here](https://www.pinecone.io/docs/data-types/metadata/).


In [5]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(cloud="aws", region="us-east-1")

Then, we initialize the index. We will be using Voyage's "voyage-4-lite" model for creating the embeddings, so we set the dimension to 1024.

In [6]:
import time

index_name = "amazon-products"
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1024,  # dimensionality of voyage-4-lite embeddings
        metric="dotproduct",
        spec=spec,
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '155',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 11 Mar 2026 17:48:08 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '51',
                                    'x-pinecone-request-latency-ms': '50',
                                    'x-pinecone-response-duration-ms': '53'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

We should see that the new Pinecone index has a total_vector_count of 0, as we haven't added any vectors yet.

## Embeddings
To get started with Voyage's embeddings, go [here](https://www.voyageai.com) to get an API key.

Now let's set up our Voyage client and demonstrate how to create an embedding using the `embed` method. To learn more about using Voyage embeddings with Claude, see [this notebook](https://github.com/anthropics/anthropic-cookbook/blob/main/third_party/VoyageAI/how_to_create_embeddings.md).

In [7]:
import voyageai

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

texts = ["Sample text 1", "Sample text 2"]

result = vo.embed(texts, model="voyage-4-lite", input_type="document")
print(result.embeddings[0])
print(result.embeddings[1])

[5.5941756727406755e-05, -0.011643669568002224, -0.017434371635317802, 0.022166555747389793, -0.02017405815422535, -0.042091552168130875, -0.029762960970401764, -0.0037359364796429873, -0.022913744673132896, -0.03885374218225479, 0.025030774995684624, 0.003393475664779544, -0.04458217695355415, 0.03437061607837677, 0.015690933912992477, -0.0410953015089035, 0.00019652582705020905, 0.012204059399664402, -0.008592654019594193, 0.026276087388396263, -0.016936246305704117, 0.06276373565196991, -0.0004942332743667066, 0.017558902502059937, 0.020921245217323303, -0.03461967781186104, -0.031381867825984955, 0.008219060488045216, -0.018430620431900024, 0.030136557295918465, 0.0166871827095747, 0.028517650440335274, 0.000860043743159622, -0.02204202674329281, -0.02141937054693699, 0.006039764266461134, -0.01581546477973461, -0.008717185817658901, -0.07720935344696045, -0.039849989116191864, 0.01793249510228634, 0.0018212690483778715, 0.022789213806390762, -0.028019525110721588, 0.01021155994385

## Uploading data to the Pinecone index

With our embedding model set up, we can now take our product descriptions, embed them, and upload the embeddings to the Pinecone index.

In [13]:
from time import sleep

from tqdm.auto import tqdm

descriptions = df["text"].tolist()
batch_size = 100  # how many embeddings we create and insert at once

for i in tqdm(range(0, len(descriptions), batch_size)):
    # find end of batch
    i_end = min(len(descriptions), i + batch_size)
    descriptions_batch = descriptions[i:i_end]
    # create embeddings (try-except added to avoid RateLimitError. Voyage currently allows 300/requests per minute.)
    done = False
    while not done:
        try:
            res = vo.embed(descriptions_batch, model="voyage-4-lite", input_type="document")
            done = True
        except Exception:
            sleep(5)

    embeds = [record for record in res.embeddings]
    # create unique IDs for each text
    ids_batch = [f"description_{idx}" for idx in range(i, i_end)]

    # Create metadata dictionaries for each text
    # Encode then decode to remove invalid Unicode surrogates
    metadata_batch = [
        {"description": description.encode("utf-8", errors="replace").decode("utf-8")}
        for description in descriptions_batch
    ]

    to_upsert = list(zip(ids_batch, embeds, metadata_batch, strict=False))

    # upsert to Pinecone
    index.upsert(vectors=to_upsert)

  0%|          | 0/101 [00:00<?, ?it/s]

## Making queries

With our index populated, we can start making queries to get results. We can take a natural language question, embed it, and query it against the index to return semantically similar product descriptions.

In [14]:
USER_QUESTION = (
    "I want to get my daughter more interested in science. What kind of gifts should I get her?"
)

question_embed = vo.embed([USER_QUESTION], model="voyage-4-lite", input_type="query")
results = index.query(vector=question_embed.embeddings, top_k=5, include_metadata=True)
results

QueryResponse(matches=[{'id': 'description_7125',
 'metadata': {'description': 'Product Name: ScienceWiz / Energy Experiment '
                             'Kit\n'
                             '\n'
                             'About Product: Join the race to save the planet '
                             'with this award winning kit that includes 22 '
                             'activities | Discover what energy is, how we '
                             'make it today and what choices await us in the '
                             "future | Winner of Dr.Toy's Top Ten Toys Award "
                             'and a Creative Child Magazine Top Choice | '
                             'Includes a 48 page book and materials for your '
                             'creations | For ages 8 to 80 - everyone will '
                             'enjoy this great kit\n'
                             '\n'
                             'Categories: Toys & Games | Learning & Education '
           

## Optimizing search

These results are good, but we can optimize them even further. Using Claude's [tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) feature, we can take the user's question and generate structured search keywords from it. This allows us to perform a wide, diverse search over the index to get more relevant product descriptions.

In [15]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def get_completion(user_message: str, system: str = "") -> str:
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": user_message}],
    )
    return message.content[0].text

In [16]:
keyword_tool = {
    "name": "generate_keywords",
    "description": "Generate diverse search keywords for a product query",
    "input_schema": {
        "type": "object",
        "properties": {
            "keywords": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of 5 diverse search keywords",
            }
        },
        "required": ["keywords"],
    },
}


def generate_keywords(question: str) -> list[str]:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=[keyword_tool],
        tool_choice={"type": "tool", "name": "generate_keywords"},
        messages=[
            {
                "role": "user",
                "content": (
                    "Given this question, generate 5 diverse search keywords "
                    f"for finding products on Amazon.\n\nQuestion: {question}"
                ),
            }
        ],
    )
    return next(block.input["keywords"] for block in response.content if block.type == "tool_use")

With our Anthropic client set up and our tool defined, we can use [tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) to generate keywords from the question. By defining a tool schema, Claude returns structured output directly — no need to parse JSON from free-form text.

In [17]:
keywords_list = generate_keywords(USER_QUESTION)
print(keywords_list)

['science kits for girls', 'STEM toys for kids', 'chemistry set for children', 'educational science experiments', 'girls science exploration gift set']


Now with our keywords in a list, let's embed each one, query it against the index, and return the top 3 most relevant product descriptions.

In [18]:
results_list = []
for keyword in keywords_list:
    # get the embeddings for the keywords
    query_embed = vo.embed([keyword], model="voyage-4-lite", input_type="query")
    # search for the embeddings in the Pinecone index
    search_results = index.query(vector=query_embed.embeddings, top_k=3, include_metadata=True)
    # append the search results to the list
    for search_result in search_results.matches:
        results_list.append(search_result["metadata"]["description"])
print(len(results_list))

15


## Answering with Claude

Now that we have a list of product descriptions, let's format them into a search template Claude has been trained with and pass the formatted descriptions into another prompt.

In [19]:
def format_results(extracted: list[str]) -> str:
    result = "\n".join(
        [
            f'<item index="{i + 1}">\n<page_content>\n{r}\n</page_content>\n</item>'
            for i, r in enumerate(extracted)
        ]
    )
    return f"\n<search_results>\n{result}\n</search_results>"

Finally, let's ask the original user's question and get our answer from Claude.

In [20]:
answer = get_completion(
    user_message=(
        f"{format_results(results_list)} Using the search results provided within "
        f"the <search_results></search_results> tags, please answer the following "
        f"question: {USER_QUESTION}. Do not reference the search results in your answer."
    ),
)
print(answer)

# Science Gift Ideas for Your Daughter

Here are some great gift ideas to spark your daughter's interest in science:

## Chemistry & Experiments
- **Chemistry sets** that let her conduct fun experiments at home, such as making bubbling "lava," changing chemical colors, or extinguishing flames with invisible gas
- **Acid & alkaline experiment kits** that teach her about mixing substances and making her own litmus paper
- **Crystal growing kits** where she can shape unique crystal designs and learn basic chemistry hands-on

## Art-Meets-Science
- **Rainbow science kits** that combine art and science, letting her create rainbow ice, rainbow butterflies, and other colorful projects — great for creative kids!

## Robotics & Engineering
- **Doodling robot kits** that she can assemble herself and watch create works of art through vibration and movement

## Physics & Electricity
- **Electric circuit experiment kits** that teach her about circuits in a fun, interactive way

## General Tips
- 🔬 